# 126 — independent SmolVLA critic-gradient reproduction (worker 0)

This is an independent implementation of the critic-gradient inference test. It deliberately does **not** call Arjun's PCP sweep or proposal drivers. It reuses only the shared SmolVLA policy, frozen critic architecture definitions, generic batched rollout engine, and Supabase schema.

Both workers together cover the exact 400 worker-83 LIBERO identities with behavior-noise stream 0. The six matched arms are: live stock; the unmodified P&P `(1,2,3)` / `K=(3,1,1)` parent; frozen-MC ascent; continued-TD ascent; frozen-MC descent; and a deterministic random direction. All nonzero controls use standardized first-10-action RMS radius `0.02`, update the clean action estimate at Euler step 3, retain 10 flow steps, and execute 10 of 50 generated actions. Exact-matched historical stock and historical `(1,2,3)` refinement columns print every 10 identities.

The notebook first requires the separately completed Jeff stock replication to match all 400 historical outcomes. This isolates failures into: base rollout reproduction, P&P parent reproduction, checkpoint loading, gradient sign, or bounded correction. Videos are off. Rerunning resumes completed rows.

In [ ]:
WORKER_INDEX = 0
WORKER_COUNT = 2
ROLLOUT_BATCH_SIZE = 8
EPISODE_LIMIT = None  # set a small integer for a diagnostic pilot; None runs this full 200-identity shard
ARM_NAMES = (
    'stock_live',
    'pnp_parent',
    'frozen_mc_ascent_r002',
    'td_ascent_r002',
    'frozen_mc_descent_r002',
    'random_r002',
)
EXPERIMENT = 'smolvla-critic-guidance-independent-jeff400-v1'
REPO_REF = 'codex/critic-guidance-repro'

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
import hashlib, json, os, zipfile
from pathlib import Path
from pnp.store import SupabaseStore

BUNDLE_SHA256 = '7e7fa08939def48654cdb00fbd0966780b4b7e087f6672d78cff653bfb252b4b'
BUNDLE_KEY = f'smolvla_pcp_sweep/bundles/{BUNDLE_SHA256}/trial_bundle.zip'
EXPECTED = {
    'frozen_mc': '9b8800bc9f27116afcb1812f93f8a9f6767ac012c7cdac790502032fbb7d0c8e',
    'cnn_mc': 'afcf832d43ffca44af843e4e5301fa0a84b1b2e485703e2a1dc13aa02b260151',
    'td': '1e3fa3f093a711d0651b69e7f152a5b56a7e7a85ec754c45b0afe5834d9f324e',
}
bundle = Path('/content') / f'critic_bundle_{BUNDLE_SHA256}.zip'
if not bundle.exists() or hashlib.sha256(bundle.read_bytes()).hexdigest() != BUNDLE_SHA256:
    payload = SupabaseStore()._download(BUNDLE_KEY)
    if hashlib.sha256(payload).hexdigest() != BUNDLE_SHA256:
        raise RuntimeError('critic bundle SHA256 mismatch')
    bundle.write_bytes(payload)
destination = Path('/content/independent-critic-checkpoints')
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in archive.namelist()):
        raise ValueError('unsafe bundle path')
    for name in archive.namelist():
        if name.startswith('checkpoints/') and name.endswith('.pt'):
            archive.extract(name, destination)
CHECKPOINT_PATHS = {name: str(destination / 'checkpoints' / f'{name}.pt') for name in EXPECTED}
for name, path in CHECKPOINT_PATHS.items():
    actual = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    if actual != EXPECTED[name]:
        raise RuntimeError(f'{name} checkpoint SHA256 mismatch: {actual}')
print({'checkpoints': CHECKPOINT_PATHS, 'sha256': EXPECTED})

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime')
from pnp.smolvla_jeff_replication import require_replication
print('Verified stock reproduction:', require_replication())
from pnp.smolvla_critic_guidance_reproduction import run_independent_critic_guidance_worker
report = run_independent_critic_guidance_worker(
    checkpoint_paths=CHECKPOINT_PATHS,
    shard_count=WORKER_COUNT,
    shard_index=WORKER_INDEX,
    arm_names=ARM_NAMES,
    rollout_batch_size=ROLLOUT_BATCH_SIZE,
    episode_limit=EPISODE_LIMIT,
    experiment=EXPERIMENT,
)
report